In [22]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import MinMaxScaler, Normalizer, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/schoch/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# https://www.kaggle.com/code/abdallahwagih/twitter-sentiment-analysis/input
df = pd.read_csv('twitterSentimentAnalysis/twitter_training.csv')
df.columns=['ID','Keyword','Sentiment','Tweet']
df.tail()

,ID,Keyword,Sentiment,Tweet
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...
74680,9200,Nvidia,Positive,Just like the windows partition of my Mac is l...


In [4]:
df.describe(include='all')

,ID,Keyword,Sentiment,Tweet
count,74681.000000,74681,74681,73995
unique,NaN,32,4,69490
top,NaN,TomClancysRainbowSix,Negative,
freq,NaN,2400,22542,172
mean,6432.640149,NaN,NaN,NaN
std,3740.423819,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN
25%,3195.000000,NaN,NaN,NaN
50%,6422.000000,NaN,NaN,NaN
75%,9601.000000,NaN,NaN,NaN


In [8]:
# Series.str ermöglicht es auf die Werte der Series als Strings zuzugreifen und Stringoperationen anzuwenden
# lower(): Umwandlung in Kleinbuchstaben
df.Tweet = df.Tweet.str.lower()

# replace()
df.Tweet = df.Tweet.str.replace(r"https?://\S+|www\.\S+", "", regex=True) # URLs entfernen
df.Tweet = df.Tweet.str.replace(r"@\w+", "", regex=True) # @ Mentions entfernen
df.Tweet = df.Tweet.str.replace(r"[^\w\s]", "", regex=True) # alles entfernen, das nicht \w - alphanumerische Zeichen oder \s beliebiges Zeichen inkl. Tab und Space ist

In [9]:
df.info()
df = df.dropna() # Zeilen mit fehlenden Werten löschen
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         74681 non-null  int64 
 1   Keyword    74681 non-null  object
 2   Sentiment  74681 non-null  object
 3   Tweet      73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 73995 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         73995 non-null  int64 
 1   Keyword    73995 non-null  object
 2   Sentiment  73995 non-null  object
 3   Tweet      73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.8+ MB


## Decision Tree und Random Forest Training und Test

In [23]:
# Dokumente (Inputvariablen) und Outputvariable trennen
X = df["Tweet"] # Input
y = df["Sentiment"] # Output

# y vorverarbeiten
encoder = LabelEncoder()
y = encoder.fit_transform(df["Sentiment"])

print(encoder.classes_)
print(y[:10])

# Hold-out-set: hierauf werden keine Modellentscheidungen getroffen
X_model, X_val, y_model, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

res = []

['Irrelevant' 'Negative' 'Neutral' 'Positive']
[3 3 3 3 3 3 3 3 3 3]


In [24]:
# Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)

Decision Tree Classifier

In [25]:
# Bag-of-Words-Vektorisierung: Vokabular auf Trainingsdaten
vectorizer = CountVectorizer(max_features=3000) 
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test) # Vektorisierung der Testdaten basierend auf dem gelernten Vokabular
print(vectorizer.get_feature_names_out())
print(vectorizer.get_feature_names_out().shape[0])

dtc = DecisionTreeClassifier(min_samples_leaf=3)
dtc.fit(X_train_bow, y_train) # y muss nicht unbedingt numerisch sein

# Vorhersagen & Bewertung
y_pred = dtc.predict(X_test_bow)
y_train_pred = dtc.predict(X_train_bow)

print(classification_report(y_test, y_pred))

print("Genauigkeit:", accuracy_score(y_test, y_pred))
print("F1 Score weighted:", f1_score(y_test, y_pred, average='weighted')) # Multiclass f1-score, mit Berücksichtigung von class imbalance

print([['Modell DTC train', f1_score(y_train, y_train_pred, average='weighted'), 'Modell DTC test', f1_score(y_test, y_pred, average='weighted')]])

['00' '000' '01' ... 'zombies' 'zone' 'zoom']
3000
              precision    recall  f1-score   support

           0       0.63      0.58      0.60      2104
           1       0.73      0.73      0.73      3619
           2       0.67      0.68      0.68      2948
           3       0.68      0.70      0.69      3169

    accuracy                           0.68     11840
   macro avg       0.68      0.67      0.67     11840
weighted avg       0.68      0.68      0.68     11840

Genauigkeit: 0.6825168918918919
F1 Score weighted: 0.6818583589431911
[['Modell DTC train', 0.8458976274941645, 'Modell DTC test', 0.6818583589431911]]


In [13]:
print(dtc.get_depth())
print(dtc.get_n_leaves())
print(dtc.max_leaf_nodes)
print(dtc.get_params())

499
7911
None
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 3, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': None, 'splitter': 'best'}


In [14]:
# Ermittlung aller ccp_alpha-Werte durch den Pruning-Pfad
path = dtc.cost_complexity_pruning_path(X_train_bow, y_train)


# Verwendung von GridSearchCV zur Bestimmung des besten ccp_alpha-Werts
param_grid = {
    "ccp_alpha": [0.0, 0.0001, 0.001, 0.005, 0.01, 0.05]
    #"max_depth": [None, 10, 100, 1000],
    #"min_samples_split": [1, 2, 3, 4]
}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=2, scoring="f1_weighted")
grid_search.fit(X_train_bow, y_train)

# Das beste ccp_alpha ermitteln
best_ccp_alpha = grid_search.best_params_['ccp_alpha']
print("Bestes ccp_alpha:", grid_search.best_params_['ccp_alpha'])

# Modell mit dem besten ccp_alpha trainieren
best_dtc = grid_search.best_estimator_
best_dtc.fit(X_train_bow, y_train)

# Vorhersagen und Genauigkeit auf den Testdaten
y_pred = best_dtc.predict(X_test_bow)
print(classification_report(y_test, y_pred))

print([['Modell DTC train', f1_score(y_train, y_train_pred, average='weighted'), 'Modell DTC test', f1_score(y_test, y_pred, average='weighted')]])

Bestes ccp_alpha: 0.0
              precision    recall  f1-score   support

  Irrelevant       0.73      0.66      0.70      2104
    Negative       0.78      0.78      0.78      3619
     Neutral       0.71      0.75      0.73      2948
    Positive       0.74      0.75      0.75      3169

    accuracy                           0.74     11840
   macro avg       0.74      0.74      0.74     11840
weighted avg       0.74      0.74      0.74     11840

[['Modell DTC train', 0.8462453825365053, 'Modell DTC test', 0.7439034462954356]]


In [15]:
# Random Forest
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_bow, y_train)

# Vorhersage
y_pred = rf.predict(X_test_bow)
y_train_pred = rf.predict(X_train_bow)

print(classification_report(y_test, y_pred))

print([['Modell RF train', f1_score(y_train, y_train_pred, average='weighted'), 'Modell RF test', f1_score(y_test, y_pred, average='weighted')]])

              precision    recall  f1-score   support

  Irrelevant       0.94      0.76      0.84      2104
    Negative       0.87      0.91      0.89      3619
     Neutral       0.85      0.86      0.86      2948
    Positive       0.84      0.89      0.86      3169

    accuracy                           0.86     11840
   macro avg       0.87      0.85      0.86     11840
weighted avg       0.87      0.86      0.86     11840

[['Modell RF train', 0.9696609381081673, 'Modell RF test', 0.8642134364934001]]


In [33]:
# Deep Learning Modell 
import tensorflow as tf

X_train_nn = X_train_bow.toarray() # achtung, max_features limitieren, toarray() kann zu Out-of-memory führen 
X_test_nn = X_test_bow.toarray()


# Neuronales Netz
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train_nn.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Training
history = model.fit(
    X_train_nn,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluation
loss, accuracy = model.evaluate(X_test_bow, y_test)

print("Test Accuracy:", accuracy)

# Vorhersagen
y_pred_probs = model.predict(X_test_nn)
y_pred = y_pred_probs.argmax(axis=1)

print(classification_report(y_test, y_pred))

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 64)             │       192,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 192,324 (751.27 KB)

 Trainable params: 192,324 (751.27 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5127 - loss: 1.1452 - val_accuracy: 0.6444 - val_loss: 0.8748
Epoch 2/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7089 - loss: 0.7449 - val_accuracy: 0.6930 - val_loss: 0.7618
Epoch 3/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7920 - loss: 0.5641 - val_accuracy: 0.7366 - val_loss: 0.6887
Epoch 4/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8435 - loss: 0.4389 - val_accuracy: 0.7562 - val_loss: 0.6464
Epoch 5/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8839 - loss: 0.3421 - val_accuracy: 0.7757 - val_loss: 0.6189
Epoch 6/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9092 - loss: 0.2697 - val_accuracy: 0.7860 - val_loss: 0.6205
Epoch 7/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9246 - loss: 0.2251 - val_accuracy: 0.7906 - val_loss: 0.6240
Epoch 8/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9376 - loss: 0.1834 - 